## Preliminaries

In [79]:
%pip install databricks-langchain
%pip install pyvis

In [80]:
import json
from databricks_langchain.chat_models import ChatDatabricks
from IPython.display import display, Markdown
import mlflow
import warnings

In [81]:
mlflow.tracing.disable()
mlflow.transformers.autolog(disable=True)
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic.main")



```
# This is formatted as code
```

## Initial Call to Model

### Set up Databricks Authentication

To interact with Databricks, you need to authenticate. The `databricks-sdk` library (used by `databricks-langchain`) can automatically pick up credentials from environment variables. We will set `DATABRICKS_HOST` and `DATABRICKS_TOKEN`.

In [96]:
import requests
from google.colab import userdata

host = userdata.get("DATABRICKS_HOST").rstrip("/")
token = userdata.get("DATABRICKS_TOKEN")

response = requests.get(
    f"{host}/api/2.0/serving-endpoints",
    headers={"Authorization": f"Bearer {token}"}
)
response.raise_for_status()

for endpoint in response.json().get("endpoints", []):
    print(endpoint["name"])

databricks-inkling
databricks-gpt-oss-120b
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-qwen35-122b-a10b
databricks-llama-4-maverick
databricks-gemma-3-12b
databricks-gte-large-en
databricks-bge-large-en
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-qwen3-embedding-0-6b


In [97]:
chat_model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b", # could use a variety of models in place of this one
    temperature=0.1, # controls the randomness of the generated text
    max_tokens=1500) # controls length of the response

In [98]:
prompt = "Write a song in the style of Celine Dion about Canada's attitude towards Donald Trump"

In [99]:
response = chat_model.invoke(prompt).content
print(response)

import json
from IPython.display import Markdown, display

response = json.loads(chat_model.invoke(prompt).content)
display(Markdown(response[-1]["text"]))


[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "We need to write a song in the style of Celine Dion about Canada's attitude towards Donald Trump. Celine Dion style: powerful ballad, emotive, soaring vocals, dramatic, perhaps with a piano/ orchestral arrangement. The lyrics should reflect Canada's attitude towards Donald Trump. Canada generally had a critical, sometimes skeptical stance, but also diplomatic. The song could be about the contrast between the two nations, the values, the tension, but also hope for unity. Use poetic language, emotional tone, maybe a chorus that repeats. Should be respectful, not hateful. The style: verses, pre-chorus, chorus, bridge. Use \"I\" perhaps as Canada personified, or narrator. Use Celine's signature \"I will love you forever\" style, but about political attitudes. Could be a ballad about \"the north wind\" etc.\n\nWrite lyrics only, maybe with some stage directions. Should be in English. Let's produce a lyrical piece.\n\nMake 

**Title: “Maple‑Leaf Whisper”**  
*(A Celine Dion‑style ballad about Canada’s quiet, hopeful stance toward the storm that was Donald Trump)*  

---

**[Intro – soft piano, gentle strings]**  

*Instrumental, a single piano arpeggio that rises like the first light over the Rockies.*  

---

**[Verse 1]**  
In the hush of the northern night,  
A maple leaf trembles in the wind,  
We hear the echo of a distant voice,  
A thunder that rolls across the plains.  

We’ve walked the road of peace for generations,  
With open doors and a gentle hand,  
When the world turned louder, we held our breath,  
And sang a softer, steadier song.  

---

**[Pre‑Chorus]**  
*The sky may darken, the storm may roar,*  
*But the north‑star keeps its steady glow.*  

---

**[Chorus – soaring, full orchestra]**  
*Oh, Canada, hear the whisper of the maple leaf,*  
*We stand with love, we stand with grace, we stand unshaken.*  
*Through the wind that rattles, through the words that sting,*  
*Our hearts beat louder, a chorus of hope.*  

*We’re not afraid to speak our truth,  
But we’ll do it with a gentle smile,  
Because the strength of a nation is not in the shout,  
It’s in the kindness that endures the miles.*  

---

**[Verse 2]**  
From the Atlantic tide to the Pacific shore,  
We watch the headlines flicker like snow,  
A leader who shouted, a world that listened,  
Yet we chose the path of quiet resolve.  

Our borders are open, our doors are wide,  
But the compass of our conscience never sways,  
We ask for respect, we ask for fairness,  
And we answer with the song of the north.  

---

**[Pre‑Chorus]**  
*When the thunder fades, the sunrise comes,*  
*And the maple leaf glows in amber light.*  

---

**[Chorus – even higher, layered vocals]**  
*Oh, Canada, hear the whisper of the maple leaf,*  
*We rise above the storm, we rise above the noise.*  
*Our voices blend like rivers meeting the sea,*  
*In harmony we find the courage to be free.*  

---

**[Bridge – half‑spoken, French verses, intimate piano]**  

*Mon cœur canadien bat au rythme du vent,*  
*Sous le ciel vaste, je trouve la paix.*  
*Nous ne crions pas, nous chantons doucement,*  
*Et la vérité danse dans la lumière.*  

*We’ll keep the promise of the north,  
A promise written in snow and song,  
To love, to listen, to stand together—  
Even when the world feels wrong.*  

---

**[Final Chorus – full choir, key change, climactic high notes]**  

*Oh, Canada, hear the whisper of the maple leaf,*  
*We are the sunrise after the night’s long storm.*  
*With every note we lift, with every breath we give,*  
*We show the world a love that’s warm.*  

*And when the wind has settled, when the echoes fade,*  
*Our song will linger, forever unafraid.*  

---

**[Outro – piano fades, a single sustained violin note]**  

*The maple leaf still whispers, soft and true,  
A promise that the north will always be you.*  

---  

*Feel the soaring vocal runs, the delicate piano interludes, and the lush orchestration that Celine Dion is known for—each line a tribute to Canada’s calm, compassionate voice amid the turbulence of a Trump‑era world.*

## Extracting Structured Representations of Unstructured Data

We see how an LLM can represent the content of news articles.  To begin, we use an article by the BBC found here: https://www.bbc.com/news/articles/cg4dzq3x3e1o

In [86]:
import requests
from bs4 import BeautifulSoup

article_url = "https://www.bbc.com/news/articles/cg4dzq3x3e1o"

response = requests.get(
    article_url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Extract the text contained in paragraphs
bbc_article = "\n".join(
    paragraph.get_text(" ", strip=True)
    for paragraph in soup.find_all("p")
)

print(bbc_article[:1000])

US President Donald Trump has imposed a 50% tariff on a wide range of goods imported from Canada, in retaliation for what he called "unequal treatment" of US cars, dairy and alcohol.
Everyday consumer items like wine and hockey sticks, as well as industrial goods such as cement, are on the target list. Several key exports, though, will be spared, including energy, potash, critical minerals and fish.
Prime Minister Mark Carney said he had spoken to Trump and both leaders agreed to "intensify" trade talks in a bid to calm growing trade tensions between the North American neighbours.
But he warned "all options" were on the table in terms of how Canada could respond to the fresh tariff threat.
Since the tariff war broke out more than 18 months ago, Carney said, Canadians have "stood together" by buying home-produced goods and agreeing new trade routes with other countries.
"[The] country is stronger, stronger because we are united, and we will remain united," he told reporters.
Tensions be

Extracting information from this raw text requires forming a prompt to the LLM.  The study of how to effectively do so is called prompt engineering.  For example, Anthropic's guide is https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/overview.

### "Be clear and direct"

In [87]:
# PRINCIPLE 1: Be clear and detailed (BAD PROMPT)
prompt1 = f"""
whom does this article talk about?:

{bbc_article}
"""

print(prompt1[:500])


whom does this article talk about?:

US President Donald Trump has imposed a 50% tariff on a wide range of goods imported from Canada, in retaliation for what he called "unequal treatment" of US cars, dairy and alcohol.
Everyday consumer items like wine and hockey sticks, as well as industrial goods such as cement, are on the target list. Several key exports, though, will be spared, including energy, potash, critical minerals and fish.
Prime Minister Mark Carney said he had spoken to Trump and 


In [88]:
chat_model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0.0, # for maximum replicability
    max_tokens=5000)

In [89]:

response = chat_model.invoke(prompt1).content
print(response)

import json
from IPython.display import Markdown, display

response = json.loads(chat_model.invoke(prompt1).content)
display(Markdown(response[-1]["text"]))

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "The user asks: \"whom does this article talk about?\" They provided a long article about US President Donald Trump, Canadian Prime Minister Mark Carney (actually Mark Carney is former Bank of Canada governor, not PM; but article says Prime Minister Mark Carney, which is inaccurate), Ontario Premier Doug Ford, US Trade Representative Jamieson Greer, etc. The main subject is about tariffs. The question: \"whom does this article talk about?\" Likely answer: It talks about US President Donald Trump, Canadian Prime Minister Mark Carney, and other officials. Probably they want the main person: Donald Trump. Could answer: The article primarily discusses US President Donald Trump and his tariff actions, also mentions Canadian Prime Minister Mark Carney, Ontario Premier Doug Ford, US Trade Representative Jamieson Greer, etc. Provide concise answer."}]}, {"type": "text", "text": "The article is centered on **U.S. President Dona

The article is primarily about **U.S. President Donald Trump** – it describes his decision to impose a 50 % tariff on a wide range of Canadian imports and the political fallout from that move.  

In the course of the story it also references several other figures who are mentioned in relation to the tariff dispute:

* **Mark Carney** – presented as Canada’s prime minister (though in reality he is a former Bank of Canada governor) and who discusses the Canadian response.  
* **Doug Ford** – Ontario premier who calls for a “tariff‑for‑tariff, dollar‑for‑dollar” retaliation.  
* **Jamieson Greer** – U.S. Trade Representative who hints at further actions.  
* **Michael Devereux** – economics professor at the University of British Columbia who comments on the impact on the USMCA.  

So the central figure is Donald Trump, with secondary mentions of Canadian officials (Mark Carney, Doug Ford) and U.S. trade officials (Jamieson Greer) and an academic commentator.

In [90]:
prompt2 = f"""
We want to extract the relevant people from a news article.

Please follow these steps:
1. Identify all the people mentioned and any description of them
2. Identify any political offices mentioned

Here is the text of the article:
{bbc_article}

"""

In [91]:
response = chat_model.invoke(prompt2).content
print(response)

import json
from IPython.display import Markdown, display

response = json.loads(chat_model.invoke(prompt2).content)
display(Markdown(response[-1]["text"]))

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "We need to extract relevant people and any description, and political offices mentioned.\n\nWe should list people with description (e.g., US President Donald Trump - President of the United States). Also mention political offices: President, Prime Minister, Premier, US Trade Representative, Supreme Court, etc.\n\nLet's parse article.\n\nPeople:\n\n- Donald Trump \u2013 US President (President of the United States)\n- Mark Carney \u2013 Prime Minister (but actually Mark Carney is former Bank of Canada governor, not PM; but article says \"Prime Minister Mark Carney\". So treat as Prime Minister of Canada per article.)\n- Jamieson Greer \u2013 US Trade Representative (U.S. Trade Representative)\n- Doug Ford \u2013 Ontario Premier (Premier of Ontario, Canada)\n- Michael Devereux \u2013 professor of economics at the University of British Columbia (academic)\n- Possibly \"the White House\" not a person.\n- No other individu

**People Mentioned (with any description given in the article)**  

| Person | Description / Role (as stated in the article) |
|--------|----------------------------------------------|
| Donald Trump | US President (President of the United States) |
| Mark Carney | Prime Minister (of Canada, according to the article) |
| Jamieson Greer | US Trade Representative |
| Doug Ford | Ontario Premier (Premier of the Canadian province of Ontario) |
| Michael Devereux | Professor of economics at the University of British Columbia |

**Political Offices Mentioned**  

| Office / Institution | Context in the article |
|----------------------|------------------------|
| President of the United States | Held by Donald Trump; the source of the tariff action |
| Prime Minister (Canada) | Held by Mark Carney (as described) |
| US Trade Representative | Held by Jamieson Greer |
| Premier of Ontario | Held by Doug Ford |
| US Supreme Court | Referenced as the body that struck down earlier tariffs |
| White House | Referenced as the executive office issuing statements and proclamations |
| USMCA (United States‑Mexico‑Canada Agreement) | Mentioned as the free‑trade agreement governing North‑American trade |
| International Emergency Economic Powers Act (IEEPA) | Cited as the law under which earlier tariffs were challenged (not a political office, but a legal authority) |
| Section 301 | Mentioned as the legal mechanism used for newer tariffs (again, a statutory authority, not an office) |

*Note: The article occasionally mis‑identifies Mark Carney as “Prime Minister” of Canada; in reality he is a former Bank of Canada governor, but the extraction follows the text as presented.*

Modern LLMs also support prompts that directly incorporate data schema which can help clarify and organize what information you want.  Some even have an explicit function mode that guarantees a particular output.

In [92]:
prompt3 = f"""
We want to extract the relevant characters from a news article.

Please provide your output in the following JSON format:

{{
  "people": [
    {{
      "name": "person's full name",
      "description": "their role or description from the article"
    }}
  ],
  "institutions": [
    {{
      "name": "institution name",
      "type": "type of institution (e.g., government, media, party, etc.)",
      "context": "brief context of how it's mentioned"
    }}
  ]
}}

Here is the text of the article:
{bbc_article}
"""

In [93]:
response = chat_model.invoke(prompt3).content
print(response)

import json
from IPython.display import Markdown, display

response = json.loads(chat_model.invoke(prompt3).content)
display(Markdown(response[-1]["text"]))

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "We need to extract people and institutions from article. Provide JSON with arrays.\n\nIdentify people: Donald Trump (US President), Mark Carney (Prime Minister? Actually Mark Carney is former Bank of Canada Governor, but article says Prime Minister Mark Carney - likely mistake; treat as Prime Minister), Jamieson Greer (US Trade Representative), Doug Ford (Ontario Premier), Michael Devereux (professor), maybe also \"the White House\" is institution, \"US Supreme Court\", \"USMCA\" is agreement not institution, \"BBC\", \"Canadian government\", \"US government\", \"University of British Columbia\". Also \"Canadian government\" mentioned.\n\nList institutions: US government (government), Canada (government), US Trade Representative (government office), USMCA (trade agreement - type: trade agreement), White House (government), Canadian government (government), BBC (media), Supreme Court (government/judicial), University o

```json
{
  "people": [
    {
      "name": "Donald Trump",
      "description": "U.S. President who imposed a 50 % tariff on a wide range of Canadian goods and announced the new duties."
    },
    {
      "name": "Mark Carney",
      "description": "Canadian Prime Minister (as described in the article) who spoke with Trump, agreed to intensify trade talks and warned all options were on the table for Canada."
    },
    {
      "name": "Jamieson Greer",
      "description": "U.S. Trade Representative who said he expected “some action soon” when asked about possible further tariffs."
    },
    {
      "name": "Doug Ford",
      "description": "Ontario Premier who warned Canada should respond “tariff for tariff, dollar for dollar” if the U.S. proceeds with the new measures."
    },
    {
      "name": "Michael Devereux",
      "description": "Professor of economics at the University of British Columbia who described the new tariffs as a significant escalation against the USMCA."
    }
  ],
  "institutions": [
    {
      "name": "United States government",
      "type": "government",
      "context": "Imposed the new 50 % tariff on Canadian goods and is pursuing a broader trade agenda."
    },
    {
      "name": "Canada government",
      "type": "government",
      "context": "Target of the U.S. tariffs and responding with potential counter‑measures and new trade routes."
    },
    {
      "name": "White House",
      "type": "government",
      "context": "Issued the tariff announcement, provided a fact sheet, and is the source of President Trump’s statements."
    },
    {
      "name": "U.S. Trade Representative (USTR) office",
      "type": "government",
      "context": "Represented by Jamieson Greer, commenting on possible future tariff actions."
    },
    {
      "name": "U.S. Supreme Court",
      "type": "judicial",
      "context": "Ruled earlier that many tariffs imposed under emergency powers were illegal, prompting the administration to seek other legal avenues."
    },
    {
      "name": "USMCA (United States‑Mexico‑Canada Agreement)",
      "type": "trade agreement",
      "context": "The existing free‑trade pact referenced to show that the new tariffs apply even to goods covered by the agreement."
    },
    {
      "name": "University of British Columbia",
      "type": "education",
      "context": "Employer of economist Michael Devereux, who provided analysis of the tariff escalation."
    },
    {
      "name": "BBC",
      "type": "media",
      "context": "News outlet that contacted the White House and the Canadian government for comment and published the article."
    },
    {
      "name": "International Emergency Economic Powers Act (IEEPA)",
      "type": "law",
      "context": "Cited as the legal basis previously used for tariffs that the Supreme Court struck down."
    },
    {
      "name": "Section 301",
      "type": "law",
      "context": "Another legal mechanism the administration is using to impose new tariffs after the IEEPA ruling."
    },
    {
      "name": "Ontario government",
      "type": "government",
      "context": "Represented by Premier Doug Ford, which warned of reciprocal tariff measures."
    }
  ]
}
```

In [94]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = """
You are a cheerful kindergarten teacher explaining the news to five-year-old
children. Describe every person and institution as if they were characters or
teams in a funny picture book. Use very simple words, short sentences, gentle
humour, and playful comparisons. Explain difficult political terms in a
child-friendly way. Do not invent facts. Return valid JSON only.
"""

prompt4 = f"""
Extract the relevant people and institutions from this news article.

Return the output in the following JSON format:

{{
  "people": [
    {{
      "name": "person's full name",
      "description": "their role or description from the article"
    }}
  ],
  "institutions": [
    {{
      "name": "institution name",
      "type": "type of institution, such as government, media, or party",
      "context": "brief context in which it is mentioned"
    }}
  ]
}}

Article:
{bbc_article}
"""

response = chat_model.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=prompt4)
])

In [95]:
import json

response = chat_model.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=prompt4)
]).content

response = json.loads(response)
result = json.loads(response[-1]["text"])

print(json.dumps(result, indent=2))

{
  "people": [
    {
      "name": "Donald Trump",
      "description": "US President who imposed a 50% tariff on many Canadian goods"
    },
    {
      "name": "Mark Carney",
      "description": "Prime Minister of Canada who spoke with Trump and called for intensified trade talks"
    },
    {
      "name": "Jamieson Greer",
      "description": "US Trade Representative who said more tariff action may be coming"
    },
    {
      "name": "Doug Ford",
      "description": "Ontario Premier who said Canada should respond tariff for tariff"
    },
    {
      "name": "Michael Devereux",
      "description": "Professor of economics at the University of British Columbia who commented on the tariffs"
    }
  ],
  "institutions": [
    {
      "name": "United States",
      "type": "government",
      "context": "Imposed new tariffs on Canadian imports"
    },
    {
      "name": "Canada",
      "type": "government",
      "context": "Target of the US tariffs and responding with its own m